In [16]:
from pathlib import Path
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

In [17]:
SEED = 42

In [18]:
DATA_DIR = Path("../data")

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

In [19]:
data = []
skipped = []

for class_folder in sorted(TRAIN_DIR.iterdir()):

    if not class_folder.is_dir():
        continue

    label = int(class_folder.name.split("_")[0])

    for img_path in class_folder.glob("*"):

        try:

            with Image.open(img_path) as img:
                img.verify()

            data.append({
                "filepath": str(img_path),
                "filename": img_path.name,
                "label": label,
                "class_name": class_folder.name
            })

        except Exception:

            skipped.append(str(img_path))

In [20]:
df = pd.DataFrame(data)

df.head()

,filepath,filename,label,class_name
0,..\data\train\0_Recyclable\R_1.jpg,R_1.jpg,0,0_Recyclable
1,..\data\train\0_Recyclable\R_10.jpg,R_10.jpg,0,0_Recyclable
2,..\data\train\0_Recyclable\R_100.jpg,R_100.jpg,0,0_Recyclable
3,..\data\train\0_Recyclable\R_1000.jpg,R_1000.jpg,0,0_Recyclable
4,..\data\train\0_Recyclable\R_1001.jpg,R_1001.jpg,0,0_Recyclable


In [21]:
print(f"Total valid images : {len(df)}")

print(f"Skipped images     : {len(skipped)}")

Total valid images : 26525
Skipped images     : 2


In [22]:
for file in skipped:
    print(file)

..\data\train\1_Electronic\1.tumpukan-e-limbah-yang-kacau-dari-laptop-dan-suku-cadang-komputer-yang-dibuang-representasi-visual-yang-luar-biasa-dari-masalah-limbah-elektronik-yang-berkembang-dan-kebutuhan-akan-solusi-daur-ulang-berkelanjutan_73523-11403.jpg
..\data\train\1_Electronic\64.dampak-lingkungan-dari-ewaste-tangkap-konsekuensi-lingkungan-dari-pembuangan-ewaste-yang-tidak-tepat-seperti-perangkat-elektronik-yang-mengeluarkan-bahan-kimia-beracun-ke-dalam-tanah-dan-saluran-air-di-lokasi-pembuangan-ilegal_997534-43151.jpg


In [23]:
df.isnull().sum()

filepath      0
filename      0
label         0
class_name    0
dtype: int64

In [24]:
print(
    "Duplicate filename:",
    df["filename"].duplicated().sum()
)

Duplicate filename: 0


In [25]:
print(
    "Duplicate filepath:",
    df["filepath"].duplicated().sum()
)

Duplicate filepath: 0


In [27]:
df["label"].value_counts().sort_index()

label
0     9999
1     3959
2    12567
Name: count, dtype: int64

In [28]:
train_df, valid_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"],
    shuffle=True
)

train_df = train_df.reset_index(drop=True)

valid_df = valid_df.reset_index(drop=True)

In [29]:
print("Train :", len(train_df))

print("Validation :", len(valid_df))

Train : 21220
Validation : 5305


In [30]:
train_df["label"].value_counts().sort_index()

label
0     7999
1     3167
2    10054
Name: count, dtype: int64

In [31]:
valid_df["label"].value_counts().sort_index()

label
0    2000
1     792
2    2513
Name: count, dtype: int64

In [32]:
train_df.to_csv(
    DATA_DIR / "train.csv",
    index=False
)

valid_df.to_csv(
    DATA_DIR / "valid.csv",
    index=False
)